# Mozix.ai — GRPO Frame Selector Training

This notebook trains the **same 513-input MLP architecture used by the Android app**. It keeps CLIP and SmolVLM2 frozen and updates only the selector.

## App compatibility contract
- Input per frame: 512-dimensional L2-normalized CLIP visual projection + one normalized RGB motion score = **513 features**.
- Frame pool: 32 uniformly sampled, square-resized frames by default.
- Inference ranking: greedy temporal diversity with `min_distance = max(1, N // 10)`, followed by a fill pass, matching `SmolVLMEngine.selectFrames()`.
- The training candidates use the same temporal-diversity rule. The app Kotlin code and its ONNX interfaces are not changed by this notebook.

Run from the `code_files/` directory after configuring the local model and MVBench paths. This notebook does not require the old `grpo_selector.py`, `query_and_baseline_clean.py`, or pHash dependency.

In [ ]:
from pathlib import Path
import json
import random
import time
import numpy as np
import torch
import components as C

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Update these paths for your machine.
MODEL_PATH = "/DATA/teaching/DL_group45/models/SmolVLM2-500M"
CLIP_PATH = "/DATA/teaching/DL_group45/models/CLIP-ViT-B-32-Local"
DATA_ROOT = Path("/DATA/teaching/DL_group45/data_set/MVBench")
ACTION_JSON = DATA_ROOT / "json/action_antonym.json"
NAV_JSON = DATA_ROOT / "json/egocentric_navigation.json"
ACTION_VIDEOS = DATA_ROOT / "video/ssv2"
NAV_VIDEOS = DATA_ROOT / "video/vlnqa"

POOL_SIZE = 32
TOP_K = 8
NUM_CANDIDATES = 8
STAGE1_EPOCHS = 2
STAGE2_EPOCHS = 3
STAGE1_LR = 1e-4
STAGE2_LR = 5e-5
CHECKPOINT_STAGE1 = Path("grpo_mlp_stage1.pt")
CHECKPOINT_FINAL = Path("grpo_mlp_final.pt")

print("Device:", C.DEVICE)
print("Selector feature dimension:", C.FEATURE_DIM)


In [ ]:
def load_mvbench_json(json_path: Path, video_folder: Path):
    if not json_path.is_file():
        raise FileNotFoundError(f"Dataset JSON not found: {json_path}")
    with json_path.open("r", encoding="utf-8") as f:
        records = json.load(f)
    if isinstance(records, dict):
        records = records.get("data", records.get("videos", []))
    if not isinstance(records, list):
        raise ValueError(f"Expected a list of MVBench samples in {json_path}")

    samples, missing = [], 0
    for item in records:
        required = ("video", "question", "candidates", "answer")
        if not all(key in item for key in required):
            continue
        video_path = video_folder / item["video"]
        if not video_path.is_file():
            missing += 1
            continue
        samples.append({
            "video_path": str(video_path),
            "question": str(item["question"]),
            "candidates": list(item["candidates"]),
            "answer": str(item["answer"]),
        })
    print(f"{json_path.name}: {len(samples)} usable, {missing} missing videos")
    return samples

action_data = load_mvbench_json(ACTION_JSON, ACTION_VIDEOS)
nav_data = load_mvbench_json(NAV_JSON, NAV_VIDEOS)
if not action_data or not nav_data:
    raise RuntimeError("One or both training datasets are empty. Check paths and MVBench files.")
print(f"Training samples: {len(action_data)} action + {len(nav_data)} navigation")


In [ ]:
# Load the frozen VLM and CLIP once. Use module attributes (C.*) below;
# do not `from components import clip_model` before load_models(), because
# that would bind the initial None value in this notebook namespace.
C.load_models(smolvlm_path=MODEL_PATH, clip_path=CLIP_PATH)
assert C.smolvlm_model is not None and C.clip_model is not None
print("SmolVLM2 and CLIP are loaded and frozen.")


In [ ]:
def extract_features_for_training(video_path: str):
    """Sample app-style frames and extract the exact (T, 513) app features."""
    sampled = C.FrameSampler.sample_frames_uniform(video_path, POOL_SIZE)
    if sampled is None:
        return None, None
    frames = sampled["frames"]
    if not frames:
        return None, None
    features = C.get_clip_embeddings_with_motion(frames)
    if features.ndim != 2 or features.shape != (len(frames), C.FEATURE_DIM):
        raise RuntimeError(f"Unexpected feature shape: {features.shape}")
    if not np.isfinite(features).all():
        return None, None
    return frames, features.astype(np.float32, copy=False)

def training_step(selector, optimizer, sample):
    frames, features = extract_features_for_training(sample["video_path"])
    if features is None or len(frames) < 2:
        return None

    frame_count = len(frames)
    k = min(TOP_K, frame_count)
    feature_tensor = torch.as_tensor(features, dtype=torch.float32, device=selector.device)
    probabilities = selector.mlp(feature_tensor)

    with torch.no_grad():
        candidates = selector.generate_candidates(
            probabilities, k=k, num_candidates=NUM_CANDIDATES
        )
    if len(candidates) < 2:
        return None

    rewards = []
    for indices in candidates:
        chosen_frames = [frames[index] for index in indices]
        try:
            prediction = C.smolvlm_answer(
                chosen_frames, sample["question"], sample["candidates"]
            )
            reward = C.compute_mcq_reward(
                prediction, sample["answer"], sample["candidates"]
            )
        except Exception as exc:
            print(f"Candidate scoring failed: {exc}")
            reward = 0.0
        rewards.append(reward)

    # A group with identical rewards has zero relative advantage and no useful update.
    if len(set(rewards)) < 2:
        return None

    optimizer.zero_grad(set_to_none=True)
    loss = C.grpo_loss(probabilities, candidates, rewards, T=frame_count)
    if not torch.isfinite(loss):
        return None
    loss.backward()
    torch.nn.utils.clip_grad_norm_(selector.mlp.parameters(), max_norm=1.0)
    optimizer.step()

    return {
        "loss": float(loss.detach().cpu()),
        "mean_reward": float(np.mean(rewards)),
        "best_reward": float(max(rewards)),
    }


In [ ]:
def train_grpo(selector, dataset, epochs, learning_rate, save_path, log_every=20):
    if not dataset:
        raise ValueError("Cannot train on an empty dataset.")
    optimizer = torch.optim.Adam(selector.mlp.parameters(), lr=learning_rate)
    selector.mlp.train()

    for epoch in range(1, epochs + 1):
        order = list(range(len(dataset)))
        random.shuffle(order)
        loss_sum = reward_sum = best_sum = 0.0
        updates = skipped = 0
        start = time.time()

        for step, sample_index in enumerate(order, start=1):
            result = training_step(selector, optimizer, dataset[sample_index])
            if result is None:
                skipped += 1
                continue
            updates += 1
            loss_sum += result["loss"]
            reward_sum += result["mean_reward"]
            best_sum += result["best_reward"]

            if step % log_every == 0:
                denominator = max(updates, 1)
                print(
                    f"Epoch {epoch}/{epochs} | {step}/{len(dataset)} | "
                    f"loss={loss_sum/denominator:.4f} | "
                    f"mean reward={reward_sum/denominator:.3f} | "
                    f"best-of-group={best_sum/denominator:.3f} | "
                    f"updates={updates} skipped={skipped}"
                )

        epoch_path = save_path.with_name(f"{save_path.stem}_ep{epoch}{save_path.suffix}")
        torch.save(selector.mlp.state_dict(), epoch_path)
        denominator = max(updates, 1)
        print(
            f"Epoch {epoch} finished in {(time.time()-start)/60:.1f} min | "
            f"updates={updates} skipped={skipped} | "
            f"loss={loss_sum/denominator:.4f} | "
            f"mean reward={reward_sum/denominator:.3f}"
        )
        if updates == 0:
            print("WARNING: no optimizer updates occurred in this epoch; inspect reward variation and dataset answers.")

    torch.save(selector.mlp.state_dict(), save_path)
    selector.mlp.eval()
    print(f"Saved final stage checkpoint: {save_path}")
    return selector


## Stage 1 — action_antonym

Train the 513-feature selector on short action-recognition clips.

In [ ]:
selector = C.GRPOFrameSelector(input_dim=C.FEATURE_DIM)
selector = train_grpo(
    selector=selector,
    dataset=action_data,
    epochs=STAGE1_EPOCHS,
    learning_rate=STAGE1_LR,
    save_path=CHECKPOINT_STAGE1,
)


## Stage 2 — navigation fine-tuning

Continue from the Stage 1 weights with a lower learning rate.

In [ ]:
selector = train_grpo(
    selector=selector,
    dataset=nav_data,
    epochs=STAGE2_EPOCHS,
    learning_rate=STAGE2_LR,
    save_path=CHECKPOINT_FINAL,
)
print("Final checkpoint:", CHECKPOINT_FINAL.resolve())


## Quick selector evaluation

This checks the trained selector in the same one-answer inference path used by the Android frame-selection policy. It is not a replacement for a full baseline/SMI benchmark comparison; those pipeline classes are not included in this repository.

In [ ]:
def evaluate_selector(selector, dataset, max_samples=100):
    selector.mlp.eval()
    subset = dataset[:max_samples]
    if not subset:
        raise ValueError("Evaluation dataset is empty.")
    correct = skipped = 0
    for index, sample in enumerate(subset, start=1):
        frames, features = extract_features_for_training(sample["video_path"])
        if features is None:
            skipped += 1
            continue
        selected = selector.select(features, k=TOP_K)
        if not selected:
            skipped += 1
            continue
        try:
            prediction = C.smolvlm_answer(
                [frames[i] for i in selected], sample["question"], sample["candidates"]
            )
            correct += C.compute_mcq_reward(
                prediction, sample["answer"], sample["candidates"]
            )
        except Exception as exc:
            skipped += 1
            print(f"Evaluation sample {index} failed: {exc}")
    evaluated = len(subset) - skipped
    accuracy = 100.0 * correct / evaluated if evaluated else float("nan")
    print(f"Accuracy: {accuracy:.2f}% ({int(correct)}/{evaluated}); skipped: {skipped}")
    return {"accuracy_percent": accuracy, "evaluated": evaluated, "skipped": skipped}

# Optional; this performs additional SmolVLM2 inference and can take time.
RUN_EVALUATION = False
if RUN_EVALUATION:
    nav_metrics = evaluate_selector(selector, nav_data, max_samples=100)
